# Day 1 Conceptual Demos

**Day 1 - Session 1**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/concepts/demos-conceptual.ipynb)

**Goal:** Explore six executable examples of task decomposition, model routing, and multi-agent integration.

Each section adapts one script from `demos-conceptual/`. Run the cells in order or run any section independently. The examples use only the Python standard library; the worktree example also requires Git, which is available in Google Colab. No API key is needed.

## Topic 1: Decomposing Complex Features

The first two demos compare task boundaries and test whether a task card contains enough durable context for independent execution.

### Demo 1: Task Boundary Check

Compare a naive feature split with a dependency-aware plan using computed file-conflict and execution-wave evidence.

In [ ]:
from collections import defaultdict

NAIVE_TASKS = [
    {"id": "email", "files": ["router.py", "email.py"], "depends_on": []},
    {"id": "sms", "files": ["router.py", "sms.py"], "depends_on": []},
    {"id": "audit", "files": ["router.py", "audit.py"], "depends_on": []},
]

BOUNDED_TASKS = [
    {"id": "schema", "files": ["schemas/shipment.json"], "depends_on": []},
    {"id": "email", "files": ["channels/email.py"], "depends_on": ["schema"]},
    {"id": "sms", "files": ["channels/sms.py"], "depends_on": ["schema"]},
    {"id": "audit", "files": ["audit/recorder.py"], "depends_on": ["schema"]},
    {
        "id": "integration",
        "files": ["router.py", "tests/test_notifications.py"],
        "depends_on": ["email", "sms", "audit"],
    },
]


def find_file_conflicts(tasks):
    owners = defaultdict(list)
    for task in tasks:
        for path in task["files"]:
            owners[path].append(task["id"])
    return {path: task_ids for path, task_ids in owners.items() if len(task_ids) > 1}


def execution_waves(tasks):
    remaining = {task["id"]: set(task["depends_on"]) for task in tasks}
    waves = []
    completed = set()
    while remaining:
        ready = sorted(task_id for task_id, deps in remaining.items() if deps <= completed)
        if not ready:
            raise ValueError("Dependency cycle detected")
        waves.append(ready)
        completed.update(ready)
        for task_id in ready:
            del remaining[task_id]
    return waves


def show_plan(name, tasks):
    conflicts = find_file_conflicts(tasks)
    print(f"\n{name}")
    print("-" * len(name))
    print(f"Tasks: {len(tasks)}")
    print(f"Shared-file conflicts: {len(conflicts)}")
    for path, owners in conflicts.items():
        print(f"  {path}: {', '.join(owners)}")
    for number, wave in enumerate(execution_waves(tasks), start=1):
        print(f"Wave {number}: {', '.join(wave)}")


print("Retail shipment-notification decomposition")
show_plan("Naive channel split", NAIVE_TASKS)
show_plan("Contract-first bounded plan", BOUNDED_TASKS)
print("\nTakeaway: Parallelize cohesive work only after contracts and hub-file ownership are explicit.")

### Demo 2: Task Card Transfer Test

Compute whether a task card replaces hidden conversational context with explicit scope, acceptance, and handoff fields.

In [ ]:
REQUIRED_FIELDS = {
    "objective",
    "deliverable",
    "context",
    "files",
    "dependencies",
    "acceptance",
    "non_goals",
    "handoff",
}
INCOMPLETE_CARD = {
    "objective": "Handle shipment email",
    "files": ["channels/email.py"],
    "acceptance": "Make sure it works",
}
TRANSFERABLE_CARD = {
    "objective": "Render email content from the approved shipment event.",
    "deliverable": "channels/email.py",
    "context": ["schemas/shipment-event.json", "tests/test_email.py"],
    "files": ["channels/email.py"],
    "dependencies": ["schema-contract approved"],
    "acceptance": "python3 -m unittest tests.test_email",
    "non_goals": ["Do not edit router.py", "Do not call a delivery vendor"],
    "handoff": ["changed files", "test result", "unresolved risks"],
}


def transfer_gaps(card):
    gaps = sorted(REQUIRED_FIELDS - card.keys())
    if card.get("acceptance") == "Make sure it works":
        gaps.append("executable acceptance command")
    return gaps


def show_card(name, card):
    gaps = transfer_gaps(card)
    print(f"\n{name}: {'NOT READY' if gaps else 'READY'}")
    for gap in gaps:
        print(f"  missing: {gap}")
    if not gaps:
        print(f"  command: {card['acceptance']}")
        print(f"  owned files: {', '.join(card['files'])}")


print("Task-card transfer test")
show_card("Conversational card", INCOMPLETE_CARD)
show_card("Durable card", TRANSFERABLE_CARD)
print("\nTakeaway: A transferable card replaces hidden conversation with scope, evidence, and handoff fields.")

## Topic 2: Selecting Models and Escalating from Evidence

The next two demos route tasks from measurable risk signals and revise that route only when execution produces actionable evidence.

### Demo 3: Model Selection Matrix

Compute a dated model, effort, and approval recommendation from task complexity, uncertainty, blast radius, verification cost, and reversibility. Model names are illustrative; the evidence-based rule is the durable part.

In [ ]:
RESEARCH_DATE = "2026-09-13"
TASKS = [
    {"name": "Update delivery copy", "complexity": 1, "uncertainty": 1, "blast_radius": 1, "verification_cost": 1, "reversible": True},
    {"name": "Add carrier mapping", "complexity": 2, "uncertainty": 1, "blast_radius": 2, "verification_cost": 1, "reversible": True},
    {"name": "Diagnose duplicate notifications", "complexity": 4, "uncertainty": 5, "blast_radius": 3, "verification_cost": 4, "reversible": True},
    {"name": "Change refund authorization", "complexity": 5, "uncertainty": 3, "blast_radius": 5, "verification_cost": 5, "reversible": False},
]


def recommend(task):
    score = sum(task[key] for key in ("complexity", "uncertainty", "blast_radius", "verification_cost"))
    if task["blast_radius"] >= 5 or not task["reversible"]:
        return score, "Claude Opus 5", "high", "required"
    if task["uncertainty"] >= 4 or score >= 14:
        return score, "Claude Opus 5", "high", "on escalation"
    if score >= 8:
        return score, "Claude Sonnet 5", "medium", "risk-based"
    return score, "Claude Haiku 4.5", "low", "not required"


def show_recommendations(tasks):
    print(f"Model-selection baseline researched {RESEARCH_DATE}")
    print("Model names are dated examples; task evidence is the durable rule.\n")
    for task in tasks:
        score, model, effort, approval = recommend(task)
        print(f"Task: {task['name']}")
        print(f"  evidence score: {score:>2} / 20")
        print(f"  initial route:  {model}, {effort} effort")
        print(f"  human approval: {approval}")


show_recommendations(TASKS)
print("Takeaway: Select model and effort from task evidence, then verify the outcome under controlled conditions.")

### Demo 4: Evidence-Based Escalation

Replay an execution trajectory to distinguish continued focused investigation, model escalation, and a mandatory human review.

In [ ]:
INITIAL_ROUTE = {"model": "Claude Sonnet 5", "effort": "high"}
TRAJECTORY = [
    {"signal": "focused_test_failed", "detail": "Duplicate event reproduced in notification router."},
    {"signal": "cross_service_scope", "detail": "Trace enters order events and customer preferences."},
    {"signal": "security_boundary", "detail": "Preference lookup affects customer authorization."},
]


def escalation_action(signal):
    actions = {
        "focused_test_failed": ("keep", "Use the exact failure to inspect the smallest relevant files."),
        "cross_service_scope": ("escalate", "Move to Claude Opus 5 at high effort for wider reasoning."),
        "security_boundary": ("human", "Pause for a human security and authorization review."),
    }
    return actions[signal]


def replay_trajectory(route, trajectory):
    print(f"Initial route: {route['model']}, {route['effort']} effort")
    for number, event in enumerate(trajectory, start=1):
        action, explanation = escalation_action(event["signal"])
        print(f"\nEvidence {number}: {event['detail']}")
        print(f"Decision: {action.upper()} - {explanation}")
        if action == "human":
            print("Execution stopped before another model attempt.")
            break


replay_trajectory(INITIAL_ROUTE, TRAJECTORY)
print("\nTakeaway: Escalate on actionable evidence, and stop when risk requires human control.")

## Topic 3: Orchestrating Multiple Agent Sessions

The final two demos isolate concurrent work in Git worktrees and admit completed work through explicit integration gates.

### Demo 5: Git Worktree Isolation

Create two real Git worktrees in a temporary repository, commit independent worker changes, and merge both branches into the integration branch.

In [ ]:
import subprocess
import tempfile
from pathlib import Path

BRANCHES = (
    ("agent/email", "email.txt", "email renderer\n"),
    ("agent/sms", "sms.txt", "sms renderer\n"),
)


def run(*args, cwd):
    return subprocess.run(args, cwd=cwd, check=True, capture_output=True, text=True).stdout.strip()


def initialize_repository(root):
    run("git", "init", "-b", "main", cwd=root)
    run("git", "config", "user.name", "Course Demo", cwd=root)
    run("git", "config", "user.email", "demo@example.invalid", cwd=root)
    (root / "README.md").write_text("shipment notification demo\n")
    run("git", "add", "README.md", cwd=root)
    run("git", "commit", "-m", "Create demo base", cwd=root)


def create_worker(root, branch, filename, content):
    worktree = root.parent / branch.replace("/", "-")
    run("git", "worktree", "add", str(worktree), "-b", branch, cwd=root)
    (worktree / filename).write_text(content)
    run("git", "add", filename, cwd=worktree)
    run("git", "commit", "-m", f"Complete {branch}", cwd=worktree)
    return worktree


def integrate(root):
    for branch, _, _ in BRANCHES:
        run("git", "merge", "--no-edit", branch, cwd=root)
    return run("git", "log", "--oneline", "--decorate", "-3", cwd=root)


with tempfile.TemporaryDirectory() as directory:
    root = Path(directory) / "notification-repo"
    root.mkdir()
    initialize_repository(root)
    workers = [create_worker(root, *definition) for definition in BRANCHES]
    print("Isolated worktrees:")
    print(run("git", "worktree", "list", cwd=root))
    print(f"\nMain sees worker files before merge: {[(root / name).exists() for _, name, _ in BRANCHES]}")
    print(f"Worker paths: {', '.join(str(path.name) for path in workers)}")
    print("\nIntegration history:")
    print(integrate(root))
    print(f"Combined files exist: {[(root / name).exists() for _, name, _ in BRANCHES]}")

print("Takeaway: Worktrees isolate edits; explicit merges and checks create the combined result.")

### Demo 6: Handoff Integration Gate

Evaluate each worker handoff against its base commit, focused check, file ownership, and unresolved-risk requirements.

In [ ]:
EXPECTED_BASE = "8f31abc"
HANDOFFS = [
    {"task": "email", "base": EXPECTED_BASE, "files": ["channels/email.py"], "check": "passed", "risk": "none"},
    {"task": "sms", "base": EXPECTED_BASE, "files": ["channels/sms.py"], "check": "passed", "risk": "none"},
]
CONFLICTING_HANDOFF = {
    "task": "sms-broad-rewrite",
    "base": EXPECTED_BASE,
    "files": ["channels/sms.py", "router.py"],
    "check": "passed",
    "risk": "shared hub changed",
}
ALLOWED_FILES = {"email": {"channels/email.py"}, "sms": {"channels/sms.py"}}


def gate(handoff):
    errors = []
    if handoff["base"] != EXPECTED_BASE:
        errors.append("base commit differs")
    if handoff["check"] != "passed":
        errors.append("focused check did not pass")
    task_key = handoff["task"].split("-")[0]
    unexpected = set(handoff["files"]) - ALLOWED_FILES.get(task_key, set())
    if unexpected:
        errors.append(f"file ownership exceeded: {', '.join(sorted(unexpected))}")
    if handoff["risk"] != "none":
        errors.append(f"unresolved risk: {handoff['risk']}")
    return errors


def show_decision(handoff):
    errors = gate(handoff)
    print(f"{handoff['task']}: {'BLOCK' if errors else 'ACCEPT'}")
    for error in errors:
        print(f"  - {error}")


print("Integration admission decisions")
for handoff in HANDOFFS:
    show_decision(handoff)
show_decision(CONFLICTING_HANDOFF)
print("Takeaway: A passing test is insufficient when the handoff violates base, ownership, or risk gates.")

## Summary

Agent orchestration is safest when task boundaries, model routes, escalation decisions, isolated workspaces, and integration gates are all derived from explicit evidence rather than hidden conversation or optimistic assumptions.